In [1]:
!pip install yfinance scikit-learn pandas numpy -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR       = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')

os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f'RAW_DIR       : {RAW_DIR}')
print(f'PROCESSED_DIR : {PROCESSED_DIR}')


RAW_DIR       : c:\Users\akbar\VSCode Project\RaksaDana\data\raw
PROCESSED_DIR : c:\Users\akbar\VSCode Project\RaksaDana\data\processed


In [3]:
tickers    = ["BBCA.JK", "BBRI.JK", "BMRI.JK"]
start_date = "2019-11-20"
end_date   = "2026-01-20"

In [4]:
raw = yf.download(tickers, start=start_date, end=end_date, progress=False)

price_data = {}
for ticker in tickers:
    df = raw.loc[:, (slice(None), ticker)].copy()
    df.columns = df.columns.droplevel(1)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index = pd.to_datetime(df.index)
    price_data[ticker] = df

In [5]:
price_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume
Date,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000


In [6]:
# Save raw price data
for ticker in tickers:
    out_path = os.path.join(RAW_DIR, f'{ticker.replace(".", "_")}_raw.csv')
    price_data[ticker].to_csv(out_path)
    print(f"Saved: {out_path}")

Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BBCA_JK_raw.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BBRI_JK_raw.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BMRI_JK_raw.csv


In [7]:
fundamental = {}

for ticker in tickers:
    info = yf.Ticker(ticker).info
    fundamental[ticker] = {
        'ROE': info.get('returnOnEquity'),
        'EPS': info.get('trailingEps'),
        'DY' : info.get('dividendYield') / 100
    }

In [8]:
fundamental

{'BBCA.JK': {'ROE': 0.22972, 'EPS': 467.26, 'DY': 0.055099999999999996},
 'BBRI.JK': {'ROE': 0.18135999, 'EPS': 389.06, 'DY': 0.134},
 'BMRI.JK': {'ROE': 0.21040002, 'EPS': 603.12, 'DY': 0.11359999999999999}}

In [9]:
merged_data = {}

for ticker in tickers:
    df = price_data[ticker].copy()
    df['ROE'] = fundamental[ticker]['ROE']
    df['EPS'] = fundamental[ticker]['EPS']
    df['DY']  = fundamental[ticker]['DY']
    merged_data[ticker] = df

In [10]:
merged_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume,ROE,EPS,DY
Date,,,,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700,0.22972,467.26,0.0551
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300,0.22972,467.26,0.0551
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300,0.22972,467.26,0.0551
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100,0.22972,467.26,0.0551
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000,0.22972,467.26,0.0551


In [11]:
def add_features(df):
    df = df.copy()
    
    df['MA7']  = df['Close'].rolling(7).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()

    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    sma20          = df['Close'].rolling(20).mean()
    std20          = df['Close'].rolling(20).std()
    df['BB_upper'] = sma20 + 2 * std20
    df['BB_lower'] = sma20 - 2 * std20
    df['BB_width'] = df['BB_upper'] - df['BB_lower']

    df['Daily_Return'] = df['Close'].pct_change()
    df['Log_Return']   = np.log(df['Close'] / df['Close'].shift(1))
    df['Volume_MA7']   = df['Volume'].rolling(7).mean()

    ema12          = df['Close'].ewm(span=12, adjust=False).mean()
    ema26          = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']         = ema12 - ema26
    df['MACD_signal']  = df['MACD'].ewm(span=9, adjust=False).mean()

    df.dropna(inplace=True)
    return df

featured_data = {}
for ticker in tickers:
    featured_data[ticker] = add_features(merged_data[ticker])

In [12]:
featured_data['BBCA.JK'].columns.tolist()

['Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'ROE',
 'EPS',
 'DY',
 'MA7',
 'MA20',
 'MA50',
 'RSI',
 'BB_upper',
 'BB_lower',
 'BB_width',
 'Daily_Return',
 'Log_Return',
 'Volume_MA7',
 'MACD',
 'MACD_signal']

In [13]:
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'MA7', 'MA20', 'MA50',
    'RSI', 'BB_upper', 'BB_lower', 'BB_width',
    'Daily_Return', 'Log_Return', 'Volume_MA7',
    'MACD', 'MACD_signal',
]

WINDOW_SIZE = 60

scaled_train_data = {}
scaled_test_data  = {}
scalers           = {}

for ticker in tickers:
    df = featured_data[ticker][feature_cols].copy()

    n_total_seq   = len(df) - WINDOW_SIZE
    n_train_seq   = int(n_total_seq * 0.8)
    train_end_idx = WINDOW_SIZE + n_train_seq

    train_df = df.iloc[:train_end_idx]
    test_df  = df.iloc[train_end_idx - WINDOW_SIZE:]

    scaler = StandardScaler()
    scaler.fit(train_df)

    scaled_train_data[ticker] = pd.DataFrame(
        scaler.transform(train_df), columns=feature_cols, index=train_df.index,
    )
    scaled_test_data[ticker] = pd.DataFrame(
        scaler.transform(test_df), columns=feature_cols, index=test_df.index,
    )
    scalers[ticker] = scaler

    print(f'{ticker:10s}  train rows {len(train_df):4d}  test rows {len(test_df):4d}')


BBCA.JK     train rows 1160  test rows  336
BBRI.JK     train rows 1160  test rows  336
BMRI.JK     train rows 1160  test rows  336


In [14]:
scaled_train_data['BBCA.JK'].tail()


,Open,High,Low,Close,Volume,MA7,MA20,MA50,RSI,BB_upper,BB_lower,BB_width,Daily_Return,Log_Return,Volume_MA7,MACD,MACD_signal
Date,,,,,,,,,,,,,,,,,
2024-11-08,1.638161,1.610148,1.587671,1.559202,0.061982,1.723984,1.868676,1.948892,-1.407683,1.931536,1.789332,0.432440,-0.640350,-0.640017,-0.416117,-1.042815,-0.574733
2024-11-11,1.392863,1.502954,1.434505,1.543903,0.691174,1.706423,1.854706,1.945388,-1.191094,1.949046,1.744937,0.741750,-0.189377,-0.182855,-0.288093,-1.265284,-0.733439
2024-11-12,1.576837,1.610149,1.557039,1.605099,-0.140983,1.682277,1.839960,1.943796,-1.339858,1.945626,1.719340,0.853985,0.573761,0.583142,-0.153847,-1.343211,-0.877193
2024-11-13,1.653493,1.625462,1.648939,1.605099,-0.441368,1.662521,1.829870,1.943477,-1.498888,1.948089,1.697164,0.977761,-0.037053,-0.029200,-0.097724,-1.392197,-1.002750
2024-11-14,1.592168,1.579521,1.587672,1.574501,-0.554178,1.627399,1.810467,1.940929,-1.737426,1.930569,1.676122,0.998084,-0.339451,-0.334613,-0.072719,-1.460545,-1.117922


In [15]:
TARGET_COL = 'Close'
TARGET_IDX = feature_cols.index(TARGET_COL)

def create_sequences(scaled_df, window_size, target_idx):
    X, y = [], []
    arr = scaled_df.values
    for i in range(window_size, len(arr)):
        X.append(arr[i - window_size : i, :])
        y.append(arr[i, target_idx])
    return np.array(X), np.array(y)

sequences = {}
for ticker in tickers:
    X_train, y_train = create_sequences(scaled_train_data[ticker], WINDOW_SIZE, TARGET_IDX)
    X_test,  y_test  = create_sequences(scaled_test_data[ticker],  WINDOW_SIZE, TARGET_IDX)
    sequences[ticker] = {
        'X_train': X_train, 'y_train': y_train,
        'X_test':  X_test,  'y_test':  y_test,
    }


In [16]:
for ticker in tickers:
    s = sequences[ticker]
    print(f"{ticker} → X_train: {s['X_train'].shape}, X_test: {s['X_test'].shape}")

BBCA.JK → X_train: (1100, 60, 17), X_test: (276, 60, 17)
BBRI.JK → X_train: (1100, 60, 17), X_test: (276, 60, 17)
BMRI.JK → X_train: (1100, 60, 17), X_test: (276, 60, 17)


In [17]:
output = {
    'sequences'    : sequences,
    'scalers'      : scalers,
    'feature_cols' : feature_cols,
    'target_col'   : TARGET_COL,
    'target_idx'   : TARGET_IDX,
    'window_size'  : WINDOW_SIZE,
    'featured_data': featured_data,
    'fundamental'  : fundamental,
    'tickers'      : tickers,
    'scaler_type'  : 'StandardScaler',
    'scaler_fit'   : 'train_only',
}

pkl_path = os.path.join(PROCESSED_DIR, 'preprocessed_data.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(output, f)
print(f'Saved: {pkl_path}')


Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\preprocessed_data.pkl


In [18]:
# Save featured CSVs
for ticker in tickers:
    out_path = os.path.join(PROCESSED_DIR, f'{ticker.replace(".", "_")}_featured.csv')
    featured_data[ticker].to_csv(out_path)
    print(f"Saved: {out_path}")

Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BBCA_JK_featured.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BBRI_JK_featured.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BMRI_JK_featured.csv
